In [3]:
import pandas as pd 
import sqlite3

In [5]:
DB_NAME = r"C:\Users\RADHAGOPINATH\recovery_revenue.db"
conn=sqlite3.connect(DB_NAME)
cursor=conn.cursor()

In [6]:
#fetch from database
cursor.execute("select * from transactions;")
rows=cursor.fetchall()

In [10]:
df=pd.DataFrame(rows,columns=["transaction_id","amount","payment_status","customer_id","event_type","failure_reason","cart_value","items_count","timestamp"])

In [12]:
df.head()

,transaction_id,amount,payment_status,customer_id,event_type,failure_reason,cart_value,items_count,timestamp
0,1,8131.83,SUCCESS,226,SUCCESSFUL_PURCHASE,None,8131.83,3,2026-06-01 17:55:55
1,2,6877.79,SUCCESS,522,SUCCESSFUL_PURCHASE,None,6877.79,4,2026-06-07 06:44:55
2,3,17608.35,SUCCESS,99,SUCCESSFUL_PURCHASE,None,17608.35,4,2026-07-01 02:47:56
3,4,18723.62,SUCCESS,959,SUCCESSFUL_PURCHASE,None,18723.62,7,2026-07-30 23:40:56
4,5,8110.44,SUCCESS,506,SUCCESSFUL_PURCHASE,None,8110.44,8,2026-07-17 18:29:52


In [13]:
df["event_type"].unique()

array(['SUCCESSFUL_PURCHASE', 'PAYMENT_FAILED', 'CHECKOUT_ABANDONED'],
      dtype=object)

In [18]:
df["event_type"].value_counts()

event_type
SUCCESSFUL_PURCHASE    2626
PAYMENT_FAILED          813
CHECKOUT_ABANDONED      561
Name: count, dtype: int64

In [21]:
df[df["event_type"]=="PAYMENT_FAILED"]

,transaction_id,amount,payment_status,customer_id,event_type,failure_reason,cart_value,items_count,timestamp
5,6,16596.41,FAILED,717,PAYMENT_FAILED,EXPIRED_CARD,16596.41,3,2026-06-20 05:03:42
6,7,12645.80,FAILED,1142,PAYMENT_FAILED,AUTHENTICATION_FAILED,12645.80,2,2026-07-15 04:11:56
7,8,5231.45,FAILED,86,PAYMENT_FAILED,EXPIRED_CARD,5231.45,2,2026-07-12 02:55:50
11,12,9125.99,FAILED,186,PAYMENT_FAILED,NETWORK_ERROR,9125.99,5,2026-06-02 09:51:11
12,13,6932.92,FAILED,1215,PAYMENT_FAILED,INSUFFICIENT_FUNDS,6932.92,5,2026-07-13 18:37:33
...,...,...,...,...,...,...,...,...,...
3990,3991,7905.63,FAILED,586,PAYMENT_FAILED,BANK_DECLINED,7905.63,2,2026-08-19 07:49:11
3992,3993,1637.51,FAILED,986,PAYMENT_FAILED,BANK_DECLINED,1637.51,6,2026-08-05 07:26:20
3996,3997,2852.50,FAILED,1367,PAYMENT_FAILED,NETWORK_ERROR,2852.50,8,2026-08-11 04:05:09
3997,3998,18102.80,FAILED,1060,PAYMENT_FAILED,EXPIRED_CARD,18102.80,7,2026-07-05 12:00:53


In [59]:
#cause analysis
def analyze_payment_cause(failure_reason):
    if failure_reason=="BANK_DECLINED":
        cause_category="BANK_RESTRICTION"
        recoverability="LOW"
    elif failure_reason=="AUTHENTICATION_FAILED":
        cause_category="CUSTOMER_ACTION_REQUIRED"
        recoverability="MEDIUM"
    elif failure_reason=="INSUFFICIENT_FUNDS":
        cause_category="CUSTOMER_FINANCIAL"
        recoverability="LOW"
    elif failure_reason=="EXPIRED_CARD":
        cause_category="CUSTOMER_ACTION_REQUIRED"
        recoverability="MEDIUM"
    elif failure_reason=="NETWORK_ERROR":
        cause_category="TEMPORARY"
        recoverability="HIGH"
    else:
        cause_category = "UNKNOWN"
        recoverability = "UNKNOWN"

    return cause_category,recoverability

    

In [ ]:
#analyze chekout abondment
def analyze_checkout(cart_value, items_count):
    # Cart value category
    if cart_value < 5000:
        value_category = "LOW_VALUE"
    elif cart_value < 15000:
        value_category = "MEDIUM_VALUE"
    else:
        value_category = "HIGH_VALUE"

    # Cart size category
    if items_count <= 2:
        cart_size = "SMALL_CART"
    elif items_count <= 5:
        cart_size = "MEDIUM_CART"
    else:
        cart_size = "LARGE_CART"

    # Recovery potential
    if value_category == "HIGH_VALUE":
        recovery_potential = "HIGH"
    elif value_category == "MEDIUM_VALUE":
        recovery_potential = "MEDIUM"
    else:
        recovery_potential = "LOW"

    return value_category, cart_size, recovery_potential


('HIGH_VALUE', 'SMALL_CART', 'HIGH')

In [75]:
def risk_level_detector(amount_at_risk):
    if amount_at_risk<5000:
                risk_level="LOW"
    elif amount_at_risk >=5000 and amount_at_risk<15000:
                risk_level="MEDIUM"
    else:
                risk_level="HIGH"
    return risk_level

results=[]
def detect_revenue_risk(event):
    for index,event in df.iterrows():
        result={}
        if event["event_type"]=="PAYMENT_FAILED":
            risk=True
            risk_type="PAYMENT_FAILURE"
            amount_at_risk=event["amount"]
            risk_level=risk_level_detector(amount_at_risk)
            cause=event["failure_reason"]
            cause_category, recoverability=analyze_payment_cause(event["failure_reason"])
            value_category = None
            cart_size = None
            recovery_potential = None

        elif event["event_type"]=="CHECKOUT_ABANDONED":
            risk=True
            risk_type="CHECKOUT_ABANDONMENT"
            amount_at_risk=event["cart_value"]
            risk_level=risk_level_detector(amount_at_risk)
            cause = None
            cause_category = None
            recoverability = None
            value_category, cart_size, recovery_potential = analyze_checkout(event["cart_value"],event["items_count"])

        else:
            risk=False
            risk_type = None
            amount_at_risk = 0
            risk_level = None
            cause=None
            cause_category=None
            recoverability=None
            value_category = None
            cart_size = None
            recovery_potential = None

        result["transaction_id"] = event["transaction_id"]
        result["risk"]=risk
        result["risk_type"]=risk_type
        result["amount_at_risk"]=amount_at_risk
        result["risk_level"]=risk_level
        result["cause"]=cause
        result["cause_category"]=cause_category
        result["recoverability"]=recoverability
        result["value_category"] = value_category
        result["cart_size"] = cart_size
        result["recovery_potential"] = recovery_potential

        results.append(result)

In [70]:
results[10:15]

[{'transaction_id': 11,
  'risk': True,
  'risk_type': 'CHECKOUT_ABANDONMENT',
  'amount_at_risk': 3552.91,
  'risk_level': 'LOW',
  'cause': None,
  'cause_category': None,
  'recoverability': None,
  'value_category': 'LOW_VALUE',
  'cart_size': 'LARGE_CART',
  'recovery_potential': 'LOW'},
 {'transaction_id': 12,
  'risk': True,
  'risk_type': 'PAYMENT_FAILURE',
  'amount_at_risk': 9125.99,
  'risk_level': 'MEDIUM',
  'cause': 'NETWORK_ERROR',
  'cause_category': 'TEMPORARY',
  'recoverability': 'HIGH',
  'value_category': None,
  'cart_size': None,
  'recovery_potential': None},
 {'transaction_id': 13,
  'risk': True,
  'risk_type': 'PAYMENT_FAILURE',
  'amount_at_risk': 6932.92,
  'risk_level': 'MEDIUM',
  'cause': 'INSUFFICIENT_FUNDS',
  'cause_category': 'CUSTOMER_FINANCIAL',
  'recoverability': 'LOW',
  'value_category': None,
  'cart_size': None,
  'recovery_potential': None},
 {'transaction_id': 14,
  'risk': True,
  'risk_type': 'CHECKOUT_ABANDONMENT',
  'amount_at_risk': 19

In [74]:
event = df.iloc[5]

detect_revenue_risk(event)

{'transaction_id': 1,
 'risk': False,
 'risk_type': None,
 'amount_at_risk': 0,
 'risk_level': None,
 'cause': None,
 'cause_category': None,
 'recoverability': None,
 'value_category': None,
 'cart_size': None,
 'recovery_potential': None}

In [47]:
risk_df=pd.DataFrame(results,columns=[
    "transaction_id",
    "risk",
    "risk_type",
    "amount_at_risk",
    "risk_level"
])

In [48]:
risk_df.head(7)

,transaction_id,risk,risk_type,amount_at_risk,risk_level
0,1,False,None,0.00,None
1,2,False,None,0.00,None
2,3,False,None,0.00,None
3,4,False,None,0.00,None
4,5,False,None,0.00,None
5,6,True,PAYMENT_FAILURE,16596.41,HIGH
6,7,True,PAYMENT_FAILURE,12645.80,MEDIUM


In [49]:
risk_df["risk"].value_counts()

risk
False    2626
True     1374
Name: count, dtype: int64

In [50]:
risk_df["risk_type"].value_counts()

risk_type
PAYMENT_FAILURE         813
CHECKOUT_ABANDONMENT    561
Name: count, dtype: int64

In [51]:
risk_df["risk_level"].value_counts()

risk_level
MEDIUM    721
HIGH      335
LOW       318
Name: count, dtype: int64

In [52]:
df["failure_reason"].value_counts()

failure_reason
BANK_DECLINED            182
EXPIRED_CARD             181
NETWORK_ERROR            158
AUTHENTICATION_FAILED    146
INSUFFICIENT_FUNDS       146
Name: count, dtype: int64

In [53]:
df.head()

,transaction_id,amount,payment_status,customer_id,event_type,failure_reason,cart_value,items_count,timestamp
0,1,8131.83,SUCCESS,226,SUCCESSFUL_PURCHASE,None,8131.83,3,2026-06-01 17:55:55
1,2,6877.79,SUCCESS,522,SUCCESSFUL_PURCHASE,None,6877.79,4,2026-06-07 06:44:55
2,3,17608.35,SUCCESS,99,SUCCESSFUL_PURCHASE,None,17608.35,4,2026-07-01 02:47:56
3,4,18723.62,SUCCESS,959,SUCCESSFUL_PURCHASE,None,18723.62,7,2026-07-30 23:40:56
4,5,8110.44,SUCCESS,506,SUCCESSFUL_PURCHASE,None,8110.44,8,2026-07-17 18:29:52


In [64]:
checkout_df = df[df["event_type"] == "CHECKOUT_ABANDONED"]
checkout_df[["cart_value", "items_count"]].describe()

,cart_value,items_count
count,561.000000,561.000000
mean,10079.563387,4.515152
std,5493.824694,2.291335
min,527.860000,1.000000
25%,5496.320000,2.000000
50%,9708.960000,5.000000
75%,14557.920000,6.000000
max,19997.180000,8.000000
